In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from utils.enums.data import Data
from utils.loss.template import ListLoss
from utils.loss.mean_ret import MeanRet
from utils.loss.max_dd import MaxDD
from utils.loss.sharpe import Sharpe
from utils.loss.sortino import Sortino
from utils.loss.ann_downside_var import AnnDownsideVar
from utils.loss.win_rate import WinRate
from utils.loss.win_loss import WinLoss
from utils.loss.var import VaR
from utils.loss.cvar import CVaR

In [2]:
# Load price data and compute returns
prices = pd.read_csv('../data/prices.csv', index_col='date')
returns = prices.pct_change().dropna().values
mean_ret = returns.mean(axis=0)

print(f'Assets: {prices.columns.tolist()}')
print(f'Return matrix shape: {returns.shape}')

Assets: ['ASSET_0', 'ASSET_1', 'ASSET_2', 'ASSET_3', 'ASSET_4']
Return matrix shape: (1259, 5)


In [3]:
# Generate random portfolios
n_assets = len(prices.columns)
n_ports = 5
np.random.seed(42)
weights = np.random.random(size=(n_ports, n_assets))
weights /= weights.sum(axis=1, keepdims=True)

print('Portfolio weights:')
pd.DataFrame(weights, columns=prices.columns, index=[f'Port_{i}' for i in range(n_ports)])

Portfolio weights:


,ASSET_0,ASSET_1,ASSET_2,ASSET_3,ASSET_4
Port_0,0.133197,0.338101,0.260318,0.212900,0.055485
Port_1,0.065285,0.024308,0.362501,0.251571,0.296334
Port_2,0.009284,0.437468,0.375464,0.095773,0.082010
Port_3,0.105673,0.175297,0.302353,0.248877,0.167800
Port_4,0.327909,0.074759,0.156568,0.196343,0.244421


In [ ]:
# Build data package
data_package = {
    Data.MeanRet: mean_ret,
    Data.Returns: returns,
}

In [5]:
# Evaluate each loss individually
losses = [
    MeanRet('Mean Return'),
    MaxDD('Max Drawdown'),
    Sharpe('Sharpe Ratio'),
    #Sortino('Sortino Ratio'),
    #AnnDownsideVar('Ann. Downside Var'),
    WinRate('Win Rate'),
    #WinLoss('Win/Loss Ratio'),
    #VaR('VaR (5%)'),
    #CVaR('CVaR (5%)'),
]

results = {}
for loss in losses:
    results[loss.name] = loss.evaluate(weights, data_package)

pd.DataFrame(results, index=[f'Port_{i}' for i in range(n_ports)]).T

/Users/baileyarm/evoport/examples/../utils/loss/max_dd.py:12: RuntimeWarning: divide by zero encountered in matmul
  port_returns = returns @ weights.T
/Users/baileyarm/evoport/examples/../utils/loss/max_dd.py:12: RuntimeWarning: overflow encountered in matmul
  port_returns = returns @ weights.T
/Users/baileyarm/evoport/examples/../utils/loss/max_dd.py:12: RuntimeWarning: invalid value encountered in matmul
  port_returns = returns @ weights.T
/Users/baileyarm/evoport/examples/../utils/loss/sharpe.py:12: RuntimeWarning: divide by zero encountered in matmul
  port_returns = returns @ weights.T
/Users/baileyarm/evoport/examples/../utils/loss/sharpe.py:12: RuntimeWarning: overflow encountered in matmul
  port_returns = returns @ weights.T
/Users/baileyarm/evoport/examples/../utils/loss/sharpe.py:12: RuntimeWarning: invalid value encountered in matmul
  port_returns = returns @ weights.T
/Users/baileyarm/evoport/examples/../utils/loss/win_rate.py:12: RuntimeWarning: divide by zero encount

,Port_0,Port_1,Port_2,Port_3,Port_4
Mean Return,0.000326,0.000164,0.000426,0.000238,0.000133
Max Drawdown,-0.234258,-0.340356,-0.222872,-0.259722,-0.265425
Sharpe Ratio,0.043543,0.014789,0.045558,0.026715,0.017765
Win Rate,0.514694,0.509134,0.516283,0.509134,0.513900


In [10]:
# Using ListLoss to evaluate all at once
list_loss = ListLoss(losses)

print('Data needed:', list_loss.data_needed())

all_results = list_loss.evaluate(weights, data_package)
df = pd.DataFrame(all_results, index=[f'Port_{i}' for i in range(n_ports)])
df.round(5)

Data needed: [<Data.Returns: 2>, <Data.MeanRet: 1>]


/Users/baileyarm/evoport/examples/../utils/loss/max_dd.py:12: RuntimeWarning: divide by zero encountered in matmul
  port_returns = returns @ weights.T
/Users/baileyarm/evoport/examples/../utils/loss/max_dd.py:12: RuntimeWarning: overflow encountered in matmul
  port_returns = returns @ weights.T
/Users/baileyarm/evoport/examples/../utils/loss/max_dd.py:12: RuntimeWarning: invalid value encountered in matmul
  port_returns = returns @ weights.T
/Users/baileyarm/evoport/examples/../utils/loss/sharpe.py:12: RuntimeWarning: divide by zero encountered in matmul
  port_returns = returns @ weights.T
/Users/baileyarm/evoport/examples/../utils/loss/sharpe.py:12: RuntimeWarning: overflow encountered in matmul
  port_returns = returns @ weights.T
/Users/baileyarm/evoport/examples/../utils/loss/sharpe.py:12: RuntimeWarning: invalid value encountered in matmul
  port_returns = returns @ weights.T
/Users/baileyarm/evoport/examples/../utils/loss/win_rate.py:12: RuntimeWarning: divide by zero encount

,Mean Return,Max Drawdown,Sharpe Ratio,Win Rate
Port_0,0.00033,-0.23426,0.04354,0.51469
Port_1,0.00016,-0.34036,0.01479,0.50913
Port_2,0.00043,-0.22287,0.04556,0.51628
Port_3,0.00024,-0.25972,0.02672,0.50913
Port_4,0.00013,-0.26542,0.01776,0.51390


In [7]:
list_loss.minimax_structure

[False, False, False, False]

In [8]:
from utils.domination import pareto as pareto

In [11]:
pareto.pareto_dominant(df = df, cols = list(df.columns), higher_is_better = list_loss.minimax_structure)

,Mean Return,Max Drawdown,Sharpe Ratio,Win Rate
Port_1,0.000164,-0.340356,0.014789,0.509134
Port_4,0.000133,-0.265425,0.017765,0.513900
